# Sistema de Rede de Cuidadores e Pacientes
**Fundamentos de Bancos de Dados 2026.1 — UFC**  
Autor: Willyam de Sousa Almeida

> Execute as células em ordem. Certifique-se de ter instalado as dependências:
> ```bash
> pip install panel psycopg2-binary sqlalchemy
> ```

In [1]:
import panel as pn
import pandas as pd
from datetime import datetime, date
from db import (
    get_session, Usuario, Cuidador, Administrador, TelefoneCuidador,
    Paciente, HistoricoMedico, Cuida, Atividade, Ocorrencia, GravidadeEnum
)

pn.extension()
print('✅ Dependências carregadas com sucesso.')

✅ Dependências carregadas com sucesso.


## Utilitários compartilhados

In [2]:
# ── Estilos globais ──────────────────────────────────────────────────────────
STYLE = """
<style>
  .card-title { font-size: 18px; font-weight: bold; color: #4a4a8a; margin-bottom: 8px; }
  .status-ok  { color: #2e7d32; font-weight: bold; }
  .status-err { color: #c62828; font-weight: bold; }
</style>
"""
pn.pane.HTML(STYLE)

def status_pane(msg, ok=True):
    cls = 'status-ok' if ok else 'status-err'
    return pn.pane.HTML(f'<p class="{cls}">{msg}</p>')

---
## 1. CRUD — Pacientes

In [3]:
# ── Widgets ──────────────────────────────────────────────────────────────────
pac_id_sel     = pn.widgets.Select(name='Selecionar paciente', options=[])
pac_nome       = pn.widgets.TextInput(name='Nome', placeholder='Nome completo')
pac_nascimento = pn.widgets.DatePicker(name='Data de Nascimento')
pac_status     = pn.pane.HTML('')
pac_tabela     = pn.pane.DataFrame(pd.DataFrame(), width=700)

def _pac_opcoes():
    s = get_session()
    try:
        rows = s.query(Paciente).order_by(Paciente.nome).all()
        return {f'{p.id_paciente} — {p.nome}': p.id_paciente for p in rows}
    finally:
        s.close()

def pac_listar(event=None):
    s = get_session()
    try:
        rows = s.query(Paciente).order_by(Paciente.nome).all()
        data = [{'ID': p.id_paciente, 'Nome': p.nome,
                 'Nascimento': p.data_nascimento} for p in rows]
        pac_tabela.object = pd.DataFrame(data)
        pac_id_sel.options = _pac_opcoes()
    finally:
        s.close()

def pac_inserir(event):
    if not pac_nome.value or not pac_nascimento.value:
        pac_status.object = '<p class="status-err">⚠️ Nome e data de nascimento são obrigatórios.</p>'
        return
    s = get_session()
    try:
        p = Paciente(nome=pac_nome.value, data_nascimento=pac_nascimento.value)
        s.add(p)
        s.commit()
        pac_status.object = '<p class="status-ok">✅ Paciente inserido com sucesso.</p>'
        pac_nome.value = ''
        pac_nascimento.value = None
        pac_listar()
    except Exception as e:
        s.rollback()
        pac_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def pac_carregar(event):
    opts = pac_id_sel.options
    if not opts:
        return
    pid = opts[pac_id_sel.value] if isinstance(opts, dict) else pac_id_sel.value
    s = get_session()
    try:
        p = s.query(Paciente).filter_by(id_paciente=pid).first()
        if p:
            pac_nome.value = p.nome
            pac_nascimento.value = p.data_nascimento
    finally:
        s.close()

def pac_editar(event):
    opts = pac_id_sel.options
    if not opts:
        pac_status.object = '<p class="status-err">⚠️ Selecione um paciente.</p>'
        return
    pid = opts[pac_id_sel.value] if isinstance(opts, dict) else pac_id_sel.value
    s = get_session()
    try:
        p = s.query(Paciente).filter_by(id_paciente=pid).first()
        if not p:
            pac_status.object = '<p class="status-err">❌ Paciente não encontrado.</p>'
            return
        p.nome = pac_nome.value or p.nome
        p.data_nascimento = pac_nascimento.value or p.data_nascimento
        s.commit()
        pac_status.object = '<p class="status-ok">✅ Paciente atualizado.</p>'
        pac_listar()
    except Exception as e:
        s.rollback()
        pac_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def pac_remover(event):
    opts = pac_id_sel.options
    if not opts:
        pac_status.object = '<p class="status-err">⚠️ Selecione um paciente.</p>'
        return
    pid = opts[pac_id_sel.value] if isinstance(opts, dict) else pac_id_sel.value
    s = get_session()
    try:
        p = s.query(Paciente).filter_by(id_paciente=pid).first()
        if p:
            s.delete(p)
            s.commit()
            pac_status.object = '<p class="status-ok">✅ Paciente removido.</p>'
            pac_listar()
    except Exception as e:
        s.rollback()
        pac_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

# Botões
btn_pac_listar  = pn.widgets.Button(name='🔄 Listar',   button_type='primary')
btn_pac_inserir = pn.widgets.Button(name='➕ Inserir',  button_type='success')
btn_pac_carregar= pn.widgets.Button(name='📥 Carregar', button_type='default')
btn_pac_editar  = pn.widgets.Button(name='✏️ Editar',   button_type='warning')
btn_pac_remover = pn.widgets.Button(name='🗑️ Remover',  button_type='danger')

btn_pac_listar.on_click(pac_listar)
btn_pac_inserir.on_click(pac_inserir)
btn_pac_carregar.on_click(pac_carregar)
btn_pac_editar.on_click(pac_editar)
btn_pac_remover.on_click(pac_remover)

pac_listar()  # carrega ao iniciar

tela_pacientes = pn.Column(
    pn.pane.Markdown('### 👤 Gerenciar Pacientes'),
    pn.Row(pac_nome, pac_nascimento),
    pn.Row(btn_pac_inserir, btn_pac_listar),
    pn.layout.Divider(),
    pn.pane.Markdown('**Selecione um paciente para editar ou remover:**'),
    pac_id_sel,
    pn.Row(btn_pac_carregar, btn_pac_editar, btn_pac_remover),
    pac_status,
    pn.layout.Divider(),
    pn.pane.Markdown('**Lista de Pacientes:**'),
    pac_tabela,
)
print('Tela Pacientes pronta.')

Tela Pacientes pronta.


---
## 2. CRUD — Cuidadores (Usuário + Cuidador + Telefone)

In [4]:
cui_id_sel  = pn.widgets.Select(name='Selecionar cuidador', options=[])
cui_nome    = pn.widgets.TextInput(name='Nome')
cui_email   = pn.widgets.TextInput(name='E-mail')
cui_senha   = pn.widgets.PasswordInput(name='Senha')
cui_cpf     = pn.widgets.TextInput(name='CPF (somente números)', max_length=11)
cui_tel     = pn.widgets.TextInput(name='Telefone principal')
cui_status  = pn.pane.HTML('')
cui_tabela  = pn.pane.DataFrame(pd.DataFrame(), width=700)

def _cui_opcoes():
    s = get_session()
    try:
        rows = s.query(Cuidador).join(Usuario).order_by(Usuario.nome).all()
        return {f'{c.id_usuario} — {c.usuario.nome}': c.id_usuario for c in rows}
    finally:
        s.close()

def cui_listar(event=None):
    s = get_session()
    try:
        rows = s.query(Cuidador).join(Usuario).order_by(Usuario.nome).all()
        data = [{'ID': c.id_usuario, 'Nome': c.usuario.nome,
                 'E-mail': c.usuario.email, 'CPF': c.cpf,
                 'Telefones': ', '.join(t.telefone for t in c.telefones)} for c in rows]
        cui_tabela.object = pd.DataFrame(data)
        cui_id_sel.options = _cui_opcoes()
    finally:
        s.close()

def cui_inserir(event):
    if not all([cui_nome.value, cui_email.value, cui_senha.value, cui_cpf.value]):
        cui_status.object = '<p class="status-err">⚠️ Nome, e-mail, senha e CPF são obrigatórios.</p>'
        return
    s = get_session()
    try:
        u = Usuario(nome=cui_nome.value, email=cui_email.value, senha=cui_senha.value)
        s.add(u)
        s.flush()
        c = Cuidador(id_usuario=u.id_usuario, cpf=cui_cpf.value)
        s.add(c)
        if cui_tel.value:
            s.add(TelefoneCuidador(id_usuario=u.id_usuario, telefone=cui_tel.value))
        s.commit()
        cui_status.object = '<p class="status-ok">✅ Cuidador inserido.</p>'
        for w in [cui_nome, cui_email, cui_senha, cui_cpf, cui_tel]:
            w.value = ''
        cui_listar()
    except Exception as e:
        s.rollback()
        cui_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def cui_carregar(event):
    opts = cui_id_sel.options
    if not opts:
        return
    cid = opts[cui_id_sel.value] if isinstance(opts, dict) else cui_id_sel.value
    s = get_session()
    try:
        c = s.query(Cuidador).filter_by(id_usuario=cid).first()
        if c:
            cui_nome.value  = c.usuario.nome
            cui_email.value = c.usuario.email
            cui_cpf.value   = c.cpf
            cui_tel.value   = c.telefones[0].telefone if c.telefones else ''
    finally:
        s.close()

def cui_editar(event):
    opts = cui_id_sel.options
    if not opts:
        cui_status.object = '<p class="status-err">⚠️ Selecione um cuidador.</p>'
        return
    cid = opts[cui_id_sel.value] if isinstance(opts, dict) else cui_id_sel.value
    s = get_session()
    try:
        c = s.query(Cuidador).filter_by(id_usuario=cid).first()
        if not c:
            return
        if cui_nome.value:  c.usuario.nome  = cui_nome.value
        if cui_email.value: c.usuario.email = cui_email.value
        if cui_cpf.value:   c.cpf           = cui_cpf.value
        if cui_tel.value and c.telefones:
            c.telefones[0].telefone = cui_tel.value
        s.commit()
        cui_status.object = '<p class="status-ok">✅ Cuidador atualizado.</p>'
        cui_listar()
    except Exception as e:
        s.rollback()
        cui_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def cui_remover(event):
    opts = cui_id_sel.options
    if not opts:
        cui_status.object = '<p class="status-err">⚠️ Selecione um cuidador.</p>'
        return
    cid = opts[cui_id_sel.value] if isinstance(opts, dict) else cui_id_sel.value
    s = get_session()
    try:
        u = s.query(Usuario).filter_by(id_usuario=cid).first()
        if u:
            s.delete(u)
            s.commit()
            cui_status.object = '<p class="status-ok">✅ Cuidador removido.</p>'
            cui_listar()
    except Exception as e:
        s.rollback()
        cui_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

btn_cui_listar  = pn.widgets.Button(name='🔄 Listar',   button_type='primary')
btn_cui_inserir = pn.widgets.Button(name='➕ Inserir',  button_type='success')
btn_cui_carregar= pn.widgets.Button(name='📥 Carregar', button_type='default')
btn_cui_editar  = pn.widgets.Button(name='✏️ Editar',   button_type='warning')
btn_cui_remover = pn.widgets.Button(name='🗑️ Remover',  button_type='danger')

btn_cui_listar.on_click(cui_listar)
btn_cui_inserir.on_click(cui_inserir)
btn_cui_carregar.on_click(cui_carregar)
btn_cui_editar.on_click(cui_editar)
btn_cui_remover.on_click(cui_remover)

cui_listar()

tela_cuidadores = pn.Column(
    pn.pane.Markdown('### 🩺 Gerenciar Cuidadores'),
    pn.Row(cui_nome, cui_email),
    pn.Row(cui_senha, cui_cpf, cui_tel),
    pn.Row(btn_cui_inserir, btn_cui_listar),
    pn.layout.Divider(),
    pn.pane.Markdown('**Selecione um cuidador para editar ou remover:**'),
    cui_id_sel,
    pn.Row(btn_cui_carregar, btn_cui_editar, btn_cui_remover),
    cui_status,
    pn.layout.Divider(),
    pn.pane.Markdown('**Lista de Cuidadores:**'),
    cui_tabela,
)
print('Tela Cuidadores pronta.')

Tela Cuidadores pronta.


---
## 3. CRUD — Atividades

In [5]:
ati_id_sel    = pn.widgets.Select(name='Selecionar atividade', options=[])
ati_descricao = pn.widgets.TextAreaInput(name='Descrição', height=80)
ati_tipo      = pn.widgets.TextInput(name='Tipo')
ati_datahora  = pn.widgets.DatetimePicker(name='Data e Hora')
ati_cuidador  = pn.widgets.Select(name='Cuidador', options=[])
ati_paciente  = pn.widgets.Select(name='Paciente', options=[])
ati_status    = pn.pane.HTML('')
ati_tabela    = pn.pane.DataFrame(pd.DataFrame(), width=700)

def _ati_opcoes_fk():
    s = get_session()
    try:
        cuis = s.query(Cuidador).join(Usuario).order_by(Usuario.nome).all()
        pacs = s.query(Paciente).order_by(Paciente.nome).all()
        ati_cuidador.options = {f'{c.id_usuario} — {c.usuario.nome}': c.id_usuario for c in cuis}
        ati_paciente.options = {f'{p.id_paciente} — {p.nome}': p.id_paciente for p in pacs}
    finally:
        s.close()

def ati_listar(event=None):
    s = get_session()
    try:
        rows = s.query(Atividade).order_by(Atividade.data_hora.desc()).all()
        data = [{'ID': a.id_atividade, 'Tipo': a.tipo,
                 'Descrição': a.descricao[:50]+'...' if len(a.descricao)>50 else a.descricao,
                 'Data/Hora': a.data_hora,
                 'Cuidador': a.cuidador.usuario.nome,
                 'Paciente': a.paciente.nome} for a in rows]
        ati_tabela.object = pd.DataFrame(data)
        ati_id_sel.options = {f'{a["ID"]} — {a["Tipo"]}': a['ID'] for a in data}
        _ati_opcoes_fk()
    finally:
        s.close()

def ati_inserir(event):
    opts_c = ati_cuidador.options
    opts_p = ati_paciente.options
    if not all([ati_descricao.value, ati_tipo.value, ati_datahora.value, opts_c, opts_p]):
        ati_status.object = '<p class="status-err">⚠️ Preencha todos os campos.</p>'
        return
    cid = opts_c[ati_cuidador.value] if isinstance(opts_c, dict) else ati_cuidador.value
    pid = opts_p[ati_paciente.value] if isinstance(opts_p, dict) else ati_paciente.value
    s = get_session()
    try:
        a = Atividade(descricao=ati_descricao.value, tipo=ati_tipo.value,
                      data_hora=ati_datahora.value, id_cuidador=cid, id_paciente=pid)
        s.add(a)
        s.commit()
        ati_status.object = '<p class="status-ok">✅ Atividade inserida.</p>'
        ati_descricao.value = ''
        ati_tipo.value = ''
        ati_listar()
    except Exception as e:
        s.rollback()
        ati_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def ati_carregar(event):
    opts = ati_id_sel.options
    if not opts:
        return
    aid = opts[ati_id_sel.value] if isinstance(opts, dict) else ati_id_sel.value
    s = get_session()
    try:
        a = s.query(Atividade).filter_by(id_atividade=aid).first()
        if a:
            ati_descricao.value = a.descricao
            ati_tipo.value = a.tipo
            ati_datahora.value = a.data_hora
    finally:
        s.close()

def ati_editar(event):
    opts = ati_id_sel.options
    if not opts:
        ati_status.object = '<p class="status-err">⚠️ Selecione uma atividade.</p>'
        return
    aid = opts[ati_id_sel.value] if isinstance(opts, dict) else ati_id_sel.value
    s = get_session()
    try:
        a = s.query(Atividade).filter_by(id_atividade=aid).first()
        if not a:
            return
        if ati_descricao.value: a.descricao = ati_descricao.value
        if ati_tipo.value:      a.tipo      = ati_tipo.value
        if ati_datahora.value:  a.data_hora = ati_datahora.value
        s.commit()
        ati_status.object = '<p class="status-ok">✅ Atividade atualizada.</p>'
        ati_listar()
    except Exception as e:
        s.rollback()
        ati_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def ati_remover(event):
    opts = ati_id_sel.options
    if not opts:
        ati_status.object = '<p class="status-err">⚠️ Selecione uma atividade.</p>'
        return
    aid = opts[ati_id_sel.value] if isinstance(opts, dict) else ati_id_sel.value
    s = get_session()
    try:
        a = s.query(Atividade).filter_by(id_atividade=aid).first()
        if a:
            s.delete(a)
            s.commit()
            ati_status.object = '<p class="status-ok">✅ Atividade removida.</p>'
            ati_listar()
    except Exception as e:
        s.rollback()
        ati_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

btn_ati_listar   = pn.widgets.Button(name='🔄 Listar',   button_type='primary')
btn_ati_inserir  = pn.widgets.Button(name='➕ Inserir',  button_type='success')
btn_ati_carregar = pn.widgets.Button(name='📥 Carregar', button_type='default')
btn_ati_editar   = pn.widgets.Button(name='✏️ Editar',   button_type='warning')
btn_ati_remover  = pn.widgets.Button(name='🗑️ Remover',  button_type='danger')

btn_ati_listar.on_click(ati_listar)
btn_ati_inserir.on_click(ati_inserir)
btn_ati_carregar.on_click(ati_carregar)
btn_ati_editar.on_click(ati_editar)
btn_ati_remover.on_click(ati_remover)

ati_listar()

tela_atividades = pn.Column(
    pn.pane.Markdown('### 📋 Gerenciar Atividades'),
    pn.Row(ati_cuidador, ati_paciente),
    pn.Row(ati_tipo, ati_datahora),
    ati_descricao,
    pn.Row(btn_ati_inserir, btn_ati_listar),
    pn.layout.Divider(),
    pn.pane.Markdown('**Selecione uma atividade para editar ou remover:**'),
    ati_id_sel,
    pn.Row(btn_ati_carregar, btn_ati_editar, btn_ati_remover),
    ati_status,
    pn.layout.Divider(),
    pn.pane.Markdown('**Lista de Atividades:**'),
    ati_tabela,
)
print('Tela Atividades pronta.')

Tela Atividades pronta.


---
## 4. CRUD — Ocorrências

In [6]:
oco_id_sel    = pn.widgets.Select(name='Selecionar ocorrência', options=[])
oco_descricao = pn.widgets.TextAreaInput(name='Descrição', height=80)
oco_gravidade = pn.widgets.Select(name='Gravidade', options=['baixa', 'media', 'alta'])
oco_datahora  = pn.widgets.DatetimePicker(name='Data e Hora')
oco_cuidador  = pn.widgets.Select(name='Cuidador', options=[])
oco_paciente  = pn.widgets.Select(name='Paciente', options=[])
oco_status    = pn.pane.HTML('')
oco_tabela    = pn.pane.DataFrame(pd.DataFrame(), width=700)

def _oco_opcoes_fk():
    s = get_session()
    try:
        cuis = s.query(Cuidador).join(Usuario).order_by(Usuario.nome).all()
        pacs = s.query(Paciente).order_by(Paciente.nome).all()
        oco_cuidador.options = {f'{c.id_usuario} — {c.usuario.nome}': c.id_usuario for c in cuis}
        oco_paciente.options = {f'{p.id_paciente} — {p.nome}': p.id_paciente for p in pacs}
    finally:
        s.close()

def oco_listar(event=None):
    s = get_session()
    try:
        rows = s.query(Ocorrencia).order_by(Ocorrencia.data_hora.desc()).all()
        data = [{'ID': o.id_ocorrencia,
                 'Gravidade': o.gravidade.value,
                 'Descrição': o.descricao[:50]+'...' if len(o.descricao)>50 else o.descricao,
                 'Data/Hora': o.data_hora,
                 'Cuidador': o.cuidador.usuario.nome,
                 'Paciente': o.paciente.nome} for o in rows]
        oco_tabela.object = pd.DataFrame(data)
        oco_id_sel.options = {f'{o["ID"]} — {o["Gravidade"]}': o['ID'] for o in data}
        _oco_opcoes_fk()
    finally:
        s.close()

def oco_inserir(event):
    opts_c = oco_cuidador.options
    opts_p = oco_paciente.options
    if not all([oco_descricao.value, oco_datahora.value, opts_c, opts_p]):
        oco_status.object = '<p class="status-err">⚠️ Preencha todos os campos.</p>'
        return
    cid = opts_c[oco_cuidador.value] if isinstance(opts_c, dict) else oco_cuidador.value
    pid = opts_p[oco_paciente.value] if isinstance(opts_p, dict) else oco_paciente.value
    s = get_session()
    try:
        o = Ocorrencia(descricao=oco_descricao.value,
                       data_hora=oco_datahora.value,
                       gravidade=GravidadeEnum[oco_gravidade.value],
                       id_cuidador=cid, id_paciente=pid)
        s.add(o)
        s.commit()
        oco_status.object = '<p class="status-ok">✅ Ocorrência inserida.</p>'
        oco_descricao.value = ''
        oco_listar()
    except Exception as e:
        s.rollback()
        oco_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def oco_carregar(event):
    opts = oco_id_sel.options
    if not opts:
        return
    oid = opts[oco_id_sel.value] if isinstance(opts, dict) else oco_id_sel.value
    s = get_session()
    try:
        o = s.query(Ocorrencia).filter_by(id_ocorrencia=oid).first()
        if o:
            oco_descricao.value = o.descricao
            oco_gravidade.value = o.gravidade.value
            oco_datahora.value  = o.data_hora
    finally:
        s.close()

def oco_editar(event):
    opts = oco_id_sel.options
    if not opts:
        oco_status.object = '<p class="status-err">⚠️ Selecione uma ocorrência.</p>'
        return
    oid = opts[oco_id_sel.value] if isinstance(opts, dict) else oco_id_sel.value
    s = get_session()
    try:
        o = s.query(Ocorrencia).filter_by(id_ocorrencia=oid).first()
        if not o:
            return
        if oco_descricao.value: o.descricao = oco_descricao.value
        if oco_gravidade.value: o.gravidade = GravidadeEnum[oco_gravidade.value]
        if oco_datahora.value:  o.data_hora = oco_datahora.value
        s.commit()
        oco_status.object = '<p class="status-ok">✅ Ocorrência atualizada.</p>'
        oco_listar()
    except Exception as e:
        s.rollback()
        oco_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

def oco_remover(event):
    opts = oco_id_sel.options
    if not opts:
        oco_status.object = '<p class="status-err">⚠️ Selecione uma ocorrência.</p>'
        return
    oid = opts[oco_id_sel.value] if isinstance(opts, dict) else oco_id_sel.value
    s = get_session()
    try:
        o = s.query(Ocorrencia).filter_by(id_ocorrencia=oid).first()
        if o:
            s.delete(o)
            s.commit()
            oco_status.object = '<p class="status-ok">✅ Ocorrência removida.</p>'
            oco_listar()
    except Exception as e:
        s.rollback()
        oco_status.object = f'<p class="status-err">❌ Erro: {e}</p>'
    finally:
        s.close()

btn_oco_listar   = pn.widgets.Button(name='🔄 Listar',   button_type='primary')
btn_oco_inserir  = pn.widgets.Button(name='➕ Inserir',  button_type='success')
btn_oco_carregar = pn.widgets.Button(name='📥 Carregar', button_type='default')
btn_oco_editar   = pn.widgets.Button(name='✏️ Editar',   button_type='warning')
btn_oco_remover  = pn.widgets.Button(name='🗑️ Remover',  button_type='danger')

btn_oco_listar.on_click(oco_listar)
btn_oco_inserir.on_click(oco_inserir)
btn_oco_carregar.on_click(oco_carregar)
btn_oco_editar.on_click(oco_editar)
btn_oco_remover.on_click(oco_remover)

oco_listar()

tela_ocorrencias = pn.Column(
    pn.pane.Markdown('### 🚨 Gerenciar Ocorrências'),
    pn.Row(oco_cuidador, oco_paciente),
    pn.Row(oco_gravidade, oco_datahora),
    oco_descricao,
    pn.Row(btn_oco_inserir, btn_oco_listar),
    pn.layout.Divider(),
    pn.pane.Markdown('**Selecione uma ocorrência para editar ou remover:**'),
    oco_id_sel,
    pn.Row(btn_oco_carregar, btn_oco_editar, btn_oco_remover),
    oco_status,
    pn.layout.Divider(),
    pn.pane.Markdown('**Lista de Ocorrências:**'),
    oco_tabela,
)
print('Tela Ocorrências pronta.')

Tela Ocorrências pronta.


---
## 5. Aplicação Principal — Tabs

In [7]:
app = pn.Tabs(
    ('👤 Pacientes',    tela_pacientes),
    ('🩺 Cuidadores',   tela_cuidadores),
    ('📋 Atividades',   tela_atividades),
    ('🚨 Ocorrências',  tela_ocorrencias),
)

pn.Column(
    pn.pane.Markdown('# 🏥 Sistema de Rede de Cuidadores e Pacientes'),
    pn.pane.Markdown('*Fundamentos de Bancos de Dados 2026.1 — UFC | Willyam de Sousa Almeida*'),
    pn.layout.Divider(),
    app,
).servable()

Traceback (most recent call last):
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyviz_comms\__init__.py", line 341, in _handle_msg
 self._on_msg(msg)
 ~~~~~~~~~~~~^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\panel\viewable.py", line 501, in _on_msg
 patch.apply_to_document(doc, comm.id if comm else None)
 ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\protocol\messages\patch_doc.py", line 102, in apply_to_document
 invoke_with_curdoc(doc, lambda: doc.apply_json_patch(self.payload, setter=setter))
 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\callbacks.py", line 462, in invoke_with_curdoc
 return f()
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\protocol\messages\patch_doc.py", line 102, in <lambda>
 invoke_with_curdoc(doc, lambda: doc.apply_json_patch(self.payload, setter=setter))
 ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\document.py", line 408, in apply_json_patch
 DocumentPatchedEvent.handle_event(self, event, setter)
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\events.py", line 245, in handle_event
 event_cls._handle_event(doc, event)
 ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\events.py", line 280, in _handle_event
 cb(event.msg_data)
 ~~^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\callbacks.py", line 409, in trigger_event
 model._trigger_event(event)
 ~~~~~~~~~~~~~~~~~~~~^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\util\callback_manager.py", line 114, in _trigger_event
 self.document.callbacks.notify_event(cast(Model, self), event, invoke)
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\callbacks.py", line 271, in notify_event
 invoke_with_curdoc(doc, callback_invoker)
 ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\document\callbacks.py", line 462, in invoke_with_curdoc
 return f()
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bokeh\util\callback_manager.py", line 110, in invoke
 cast(EventCallbackWithEvent, callback)(event)
 ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
 File "C:\Users\willy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\panel\reactive.py", line 600, in _comm_event
 state._handle_exception(e)
 ~~~~~~~~~~~~~~~~~~~~

Column
    [0] Markdown(str)
    [1] Markdown(str)
    [2] Divider()
    [3] Tabs
        [0] Column
            [0] Markdown(str)
            [1] Row
                [0] TextInput(label='Nome', name='Nome', placeholder='Nome completo')
                [1] DatePicker(label='Data de Nascimento', name='Data de Nascimento')
            [2] Row
                [0] Button(button_type='success', color='success', label='➕ Inserir', name='➕ Inserir')
                [1] Button(button_type='primary', color='primary', label='🔄 Listar', name='🔄 Listar')
            [3] Divider()
            [4] Markdown(str)
            [5] Select(label='Selecionar paciente', name='Selecionar paciente', options={'8 — Antônio Ferreira': 8...}, value=8)
            [6] Row
                [0] Button(label='📥 Carregar', name='📥 Carregar')
                [1] Button(button_type='warning', color='warning', label='✏️ Editar', name='✏️ Editar')
                [2] Button(button_type='danger', color='danger', label='🗑️ Remover', name='🗑️ Remover')
            [7] HTML(str)
            [8] Divider()
            [9] Markdown(str)
            [10] DataFrame(DataFrame, width=700)
        [1] Column
            [0] Markdown(str)
            [1] Row
                [0] TextInput(label='Nome', name='Nome')
                [1] TextInput(label='E-mail', name='E-mail')
            [2] Row
                [0] PasswordInput(label='Senha', name='Senha')
                [1] TextInput(label='CPF (somente números)', max_length=11, name='CPF (somente números)')
                [2] TextInput(label='Telefone principal', name='Telefone principal')
            [3] Row
                [0] Button(button_type='success', color='success', label='➕ Inserir', name='➕ Inserir')
                [1] Button(button_type='primary', color='primary', label='🔄 Listar', name='🔄 Listar')
            [4] Divider()
            [5] Markdown(str)
            [6] Select(label='Selecionar cuidador', name='Selecionar cuidador', options={'1 — Ana Lima': 1, ...}, value=1)
            [7] Row
                [0] Button(label='📥 Carregar', name='📥 Carregar')
                [1] Button(button_type='warning', color='warning', label='✏️ Editar', name='✏️ Editar')
                [2] Button(button_type='danger', color='danger', label='🗑️ Remover', name='🗑️ Remover')
            [8] HTML(str)
            [9] Divider()
            [10] Markdown(str)
            [11] DataFrame(DataFrame, width=700)
        [2] Column
            [0] Markdown(str)
            [1] Row
                [0] Select(label='Cuidador', name='Cuidador', options={'1 — Ana Lima': 1, ...}, value=1)
                [1] Select(label='Paciente', name='Paciente', options={'8 — Antônio Ferreira': 8...}, value=8)
            [2] Row
                [0] TextInput(label='Tipo', name='Tipo')
                [1] DatetimePicker(as_numpy_datetime64=False, label='Data e Hora', name='Data e Hora')
            [3] TextAreaInput(height=80, label='Descrição', name='Descrição')
            [4] Row
                [0] Button(button_type='success', color='success', label='➕ Inserir', name='➕ Inserir')
                [1] Button(button_type='primary', color='primary', label='🔄 Listar', name='🔄 Listar')
            [5] Divider()
            [6] Markdown(str)
            [7] Select(label='Selecionar atividade', name='Selecionar atividade', options={'11 — Recreação': 11, ...}, value=11)
            [8] Row
                [0] Button(label='📥 Carregar', name='📥 Carregar')
                [1] Button(button_type='warning', color='warning', label='✏️ Editar', name='✏️ Editar')
                [2] Button(button_type='danger', color='danger', label='🗑️ Remover', name='🗑️ Remover')
            [9] HTML(str)
            [10] Divider()
            [11] Markdown(str)
            [12] DataFrame(DataFrame, width=700)
        [3] Column
            [0] Markdown(str)
            [1] Row
                [0] Select(label='Cuidador', name='Cuidador', options={'1 — Ana Lima': 1, ...}, val